# Mini-TP 2 — Metadatos del modelo por GraphQL (Sesión 2)

Expone los metadatos del modelo del Mini-TP 1 por **GraphQL** (FastAPI +
Strawberry) y **compara** la misma lectura contra REST: cuántas llamadas y
cuántos bytes cuesta armar *la vista del modelo* (`name`, `version`, `auc`,
`accuracy`, `f1`) con cada enfoque.

Código: `src/model_service/graphql.py` y `src/model_service/rest.py`
(mismo `model.py` por debajo).

## 1. Levantar las dos APIs en segundo plano

In [1]:
import threading, time, socket, urllib.request, uvicorn

def serve(app, port):
    def _run():
        uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port,
                                      log_level="warning")).run()
    threading.Thread(target=_run, daemon=True).start()
    for _ in range(50):
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"el servidor en :{port} no respondió")

from model_service.rest import app as rest_app
from model_service.graphql import app as gql_app

serve(rest_app, 8000)
serve(gql_app, 8010)
REST = "http://127.0.0.1:8000"
GQL = "http://127.0.0.1:8010/graphql"
print("REST   ->", REST)
print("GraphQL->", GQL)

REST   -> http://127.0.0.1:8000
GraphQL-> http://127.0.0.1:8010/graphql


## 2. La query GraphQL — una sola llamada, sólo los campos pedidos

In [2]:
import requests
QUERY = "{ model { name version metrics { auc accuracy f1 } } }"
rg = requests.post(GQL, json={"query": QUERY})
print("HTTP", rg.status_code)
rg.json()

HTTP 200


{'data': {'model': {'name': 'RandomForestClassifier',
   'version': 1,
   'metrics': {'auc': 0.9933, 'accuracy': 0.9, 'f1': 0.8997}}}}

## 3. Cliente Python: armar la misma vista por REST

In [3]:
# REST no tiene un endpoint "vista del modelo": hay que componer 2 llamadas
r_info = requests.get(f"{REST}/v1/model")             # name, version (+ campos de más)
r_metrics = requests.get(f"{REST}/v1/model/metrics")  # auc, accuracy, f1

vista_rest = {
    "name": r_info.json()["name"],
    "version": r_info.json()["model_version"],
    "metrics": r_metrics.json(),
}
vista_rest

{'name': 'RandomForestClassifier',
 'version': '1.0.0',
 'metrics': {'accuracy': 0.9, 'auc': 0.9933, 'f1': 0.8997}}

## 4. Medición: llamadas y bytes

In [4]:
rest_calls = 2
rest_bytes = len(r_info.content) + len(r_metrics.content)

gql_calls = 1
gql_bytes = len(rg.content)

info = r_info.json()
campos_traidos = len(info)
campos_usados = 2

print(f"{'':12} {'llamadas':>10} {'bytes':>10}")
print(f"{'REST':12} {rest_calls:>10} {rest_bytes:>10}")
print(f"{'GraphQL':12} {gql_calls:>10} {gql_bytes:>10}")
print()
print(f"REST /v1/model: {campos_traidos} campos traídos, {campos_usados} usados "
      f"-> over-fetching de {campos_traidos - campos_usados} campos")

               llamadas      bytes
REST                  2        319
GraphQL               1        128

REST /v1/model: 6 campos traídos, 2 usados -> over-fetching de 4 campos


## 5. Conclusión — REST vs GraphQL

|  | Llamadas | Campos de más |
|---|---|---|
| **REST** | 2 (`/v1/model` + `/v1/model/metrics`) | sí: `/v1/model` trae `dataset`, `features`, `target_names`, `trained_at` |
| **GraphQL** | 1 | no |

- **Under-fetching en REST:** ningún endpoint devuelve *la vista* completa, así
  que el cliente **compone** varias llamadas (2 acá; con recursos anidados
  aparecería el problema **N+1**).
- **Over-fetching en REST:** `/v1/model` devuelve todos los metadatos aunque la
  vista sólo use `name` y `version`.
- **GraphQL:** una request con la forma exacta del dato. El cliente declara qué
  campos quiere y el servidor responde eso y nada más.
- **Costo de GraphQL:** hay que definir esquema y resolvers (más código y una
  capa conceptual extra). Para una API chica y estable, REST alcanza; GraphQL
  gana con muchos consumidores de necesidades distintas o grafos de datos con
  relaciones (linaje, MLflow).